In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / "src").is_dir() and (candidate / "project.ipynb").is_file():
        ROOT = candidate
        break
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from pyspark.sql import SparkSession
from src.dataOI import DataOI

from src.seed import SEED

# True: rebuild inputs and the selected embeddings; False: load their saved frames.
# False: skip that work and load the saved train, validation, and test embedding frames from data/processed/embeddings.
REBUILD = False

# `spark.driver.memory` is read only when the JVM starts. Restart the kernel after changing it.
spark = SparkSession.builder \
                            .master("local[*]")\
                            .appName("DrugReviewClassification")\
                            .config("spark.driver.memory","16g")\
                            .getOrCreate()
spark_context = spark.sparkContext
spark_context.setLogLevel("ERROR")
data_oi = DataOI()


In [ ]:
from src.cleaning import data_cleaning
from src.text_features import text_feature_engineering
from src.encoder import encoder
from src.split import split, weight

if REBUILD:
    df_raw = data_oi.load_raw_data()
    df_cleaned = data_cleaning(df_raw).cache()
    df_cleaned.count()
    df_engineered = text_feature_engineering(df_cleaned, True).cache()
    df_engineered.count()
    split_frames = split(df_engineered, SEED)
    df_weighted = weight(split_frames[0])
    df_train, df_valid, df_test = encoder(df_weighted, *split_frames[1:])
    for frame in (df_train, df_valid, df_test):
        frame.count()
    data_oi.save_split_data(df_train, df_valid, df_test)
    data_oi.unpersist(df_engineered, df_weighted, *split_frames)
    del df_engineered, df_weighted, split_frames, frame


In [ ]:
from src.embeddings import EmbeddingModel
from src.text_features import baseline_text_feature_engineering, fit_phrases

# TF-IDF is an optional diagnostic input; it does not count as an advanced embedding.
USE_EMBEDDINGS = {"tfidf": True, "glove": False}
embedding_names = [name for name, enabled in USE_EMBEDDINGS.items() if enabled]

if REBUILD:
    with data_oi.scope():
        phrases = fit_phrases(df_train) if "glove" in embedding_names else None
        embedding_model = EmbeddingModel(df_train, df_valid, df_test, phrases)
        for name in embedding_names:
            print(f"Embedding Model `{name}` Training")
            if name == "tfidf":
                # Keep the shared tfidf artifact on the tokenizer-only baseline path.
                baseline_text = baseline_text_feature_engineering(df_cleaned).cache()
                baseline_text.count()
                baseline_parts = split(baseline_text, SEED)
                baseline_weighted = weight(baseline_parts[0])
                baseline_frames = encoder(baseline_weighted, *baseline_parts[1:])
                baseline_model = EmbeddingModel(*baseline_frames)
                baseline_model.baseline_train(name)
                embedding_frames = baseline_model.baseline_predict(name)
            else:
                EmbeddingModel.train_registry[name](embedding_model)
                embedding_frames = EmbeddingModel.predict_registry[name](embedding_model)
            data_oi.save_embedding_data(name, *embedding_frames)
    data_oi.unpersist(df_cleaned, df_train, df_valid, df_test)
    del df_raw, df_cleaned, df_train, df_valid, df_test, embedding_model

train_embedding_dict = {}
valid_embedding_dict = {}
test_embedding_dict = {}
for name in embedding_names:
    train_embedding_dict[name], valid_embedding_dict[name], test_embedding_dict[name] = (
        data_oi.load_embedding_data(name)
    )
print(f"Embeddings in this run: {embedding_names}")


In [ ]:
from src.analytics import AnalyticsModel

USE_MODELS = {"svm": True}
para = {}  # Flat overrides for the current SVM constructor parameters.
unknown = set(USE_MODELS) - set(AnalyticsModel.registry)
if unknown:
    raise KeyError(f"USE_MODELS names {sorted(unknown)} not in registry {sorted(AnalyticsModel.registry)}")
analytics_names = [name for name, enabled in USE_MODELS.items() if enabled]
print(f"Analytics models in this run: {analytics_names}")

analytics_models = {
    embedding_name: AnalyticsModel(
        embedding_name,
        train_embedding_dict[embedding_name],
        valid_embedding_dict[embedding_name],
        test_embedding_dict[embedding_name],
    )
    for embedding_name in embedding_names
}


In [ ]:
run_names = []
for embedding_name in embedding_names:
    model = analytics_models[embedding_name]
    try:
        for analytics_name in analytics_names:
            print(f"Embedding `{embedding_name}` Analytics `{analytics_name}` Running")
            model_name = f"{analytics_name}__{embedding_name}"
            try:
                predictions = AnalyticsModel.registry[analytics_name](model, para=para)
                data_oi.save_prediction_data(model_name, *predictions)
                run_names.append(model_name)
                del predictions
            finally:
                model.release_features()
    finally:
        data_oi.unpersist(
            train_embedding_dict[embedding_name],
            valid_embedding_dict[embedding_name],
            test_embedding_dict[embedding_name],
        )
del analytics_models


In [ ]:
from src.evaluation import Scorer

USE_BASELINE_LR = False
score_names = run_names + (["lr__tfidf"] if USE_BASELINE_LR else [])
train_pred_dict, valid_pred_dict, test_pred_dict = data_oi.load_prediction_data(score_names)

print("Training Data:")
Scorer(train_pred_dict).show_metrics(truncate=80)
print("Validation Data:")
Scorer(valid_pred_dict).show_metrics(truncate=80)
print("Testing Data:")
Scorer(test_pred_dict).show_metrics(truncate=80)
